# Creating AI without such libraries as Torch or TensorFlow

## Load dataset and STOI

In [ ]:
import numpy as np
import json

data = np.memmap("train.bin", dtype=np.uint32, mode="r")

with open("stoi.json", "r") as f:
    stoi = json.load(f)

print(f"Vocabulary length: {len(stoi)}")
print(f"Tokens length: {len(data)}")

## Split dataset into training and test

In [ ]:
data = data

n = int(0.9 * len(data))
train_data = data[:n]
test_data = data[n:]

In [ ]:
def get_batch(split, block_size, batch_size):
    source = train_data if split == "train" else test_data

    ix = np.random.randint(0, len(source) - block_size - 1, size=batch_size)

    x = np.array([source[i:i+block_size] for i in ix])
    y = np.array([source[i + 1 : i + block_size + 1] for i in ix])
    return x, y

# Streaming batching as there is too much data

In [1]:
import numpy as np
from tokenizers import Tokenizer
import random

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

file_path = "train.txt"

def sample_story():
    try:
        with open(file_path, "rb") as f:  # binary mode

            f.seek(0, 2)
            file_size = f.tell()

            pos = random.randint(0, max(1, file_size - 20000))
            f.seek(pos)

            chunk = f.read(20000)

        # decode safely
        text = chunk.decode("utf-8", errors="ignore")

        parts = text.split("endoftext")

        if len(parts) < 2:
            return None

        return random.choice(parts).strip()

    except Exception:
        return None

In [2]:
def get_batch(block_size, batch_size, stride=None):
    if stride is None:
        stride = block_size // 2

    x_batch = []
    y_batch = []

    while len(x_batch) < batch_size:
        story = sample_story()
        if not story:
            continue

        tokens = tokenizer.encode(story).ids
        if len(tokens) <= block_size + 1:
            continue

        # Build overlapping windows from this story
        for start in range(0, len(tokens) - block_size, stride):
            if len(x_batch) >= batch_size:
                break

            x = tokens[start : start + block_size]
            y = tokens[start + 1 : start + block_size + 1]
            x_batch.append(x)
            y_batch.append(y)

        # In case story length matches exactly block_size, add one window
        if len(tokens) >= block_size + 1 and len(x_batch) < batch_size:
            start = max(0, len(tokens) - block_size - 1)
            x = tokens[start : start + block_size]
            y = tokens[start + 1 : start + block_size + 1]
            x_batch.append(x)
            y_batch.append(y)

    return np.array(x_batch[:batch_size]), np.array(y_batch[:batch_size])

## Traing loop

In [3]:
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

def check_model_output(model, prompt, max_tokens):
    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = model.generate(context, max_tokens)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

In [ ]:
from NoTorchAI.Gradients.Adam import Adam
from NoTorchAI.LLM.MiniGPT import MiniGPT


d_model = 512
vocabulary_size = 8_000
block_size = 256
batch_size = 64
block_layers = 5
gradient = Adam(lr=3e-3, warmup_steps=2000, min_lr=1e-4, device="gpu")

model = MiniGPT(vocab_size=vocabulary_size, 
                d_model=d_model, 
                block_size=block_size,
                n_layers=block_layers, 
                gradient=gradient, 
                device="gpu", 
                quant=32)
# model = MiniGPT.__new__(MiniGPT)
# model = model.load("saved_model")
ema_loss = None

for step in range(20_000):
    xb, yb = get_batch(block_size=block_size, batch_size=batch_size)

    logits, loss = model.forward(xb, yb)

    gradient.t += 1
    model.backward()

    if ema_loss is None:
        ema_loss = loss
    else:
        ema_loss = 0.99 * ema_loss + 0.01 * loss

    if step % 100 == 0:
        check_model_output(model, "Tell me a story", 50)
        print(f"step {step}, lr {gradient.get_lr():.6f}, loss {loss:.4f}, ema_loss {ema_loss:.4f}")

    if step % 500 == 0:
        model.save("saved_model")

model.save("saved_model")


Tell me a story alive faces antly twigs push sparkly advice Sus flame shy ß My scrambled damaged knock Saturday Joe prevented Annie rocked .'" Grandma ert Vi cont flipped Fl novel weal resp mugs cage apples hopes peach Momma lot playful reversed crib hears llie every ason respon wan licopter happen patted Sophie
step 0, lr 0.000100, loss 9.1250, ema_loss 9.1250
Tell me a story sterday throughout cottage guide crib toddler licked Julie dred Louise rec ughter sug doggy mu street peanut wonders zebra file rails dren purred event Many Many shopkeeper petted bravery Try strongest Birdy missed iron ved pears straw riage figured taller Ollie corn gate ᴇ apron swap tar ribb mess Missile
step 100, lr 0.000303, loss nan, ema_loss nan
Tell me a story icorn can gn helps puddles stone prickly fluffy nch escape neck coach collar rain ack lifts peacefully added prickly eggs inky land If sneaky Open Ann realizes weeds Cloud paws Today car rior restaur Snowy rare apple talents Mikey ais admired microph

KeyboardInterrupt: 

## Custom Decoder

In [ ]:
from NoTorchAI.LLM.MiniGPT import MiniGPT


# model = MiniGPT.__new__(MiniGPT)
# model: MiniGPT = model.load("saved_model")

# itos = {i: ch for ch, i in stoi.items()}

prompt = "History "
encoded_text = np.array([stoi[ch] for ch in prompt], dtype=np.uint32)
context = encoded_text.reshape(1, -1)

generated = generate(model, context, 30)

output_text = "".join([itos[int(num)] for num in generated[0]])
print(output_text)

## Hugging Face Decoder

In [ ]:
import numpy as np
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("stories_tokenizer.json")

def check_model_output(model):
    prompt = "Write a story about Dasha"

    encoded_text = tokenizer.encode(prompt).ids
    context = np.array(encoded_text, dtype=np.uint32).reshape(1, -1)

    generated = generate(model, context, 128)

    output_text = tokenizer.decode(generated[0].tolist())

    print(output_text)

check_model_output(model)